## Day 4: Tree-Based Volatility Models

This notebook implements:
1) Decision Tree, Random Forest, and Gradient Boosting models,
2) light validation-based tuning,
3) comparison to Day 3 baselines/linear,
4) feature importance and model comparison plots.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.tree import DecisionTreeRegressor


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name.startswith("notebooks") else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data:"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "data"

FIG_DIR = PROJECT_ROOT / "figures:"
if not FIG_DIR.exists():
    FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DAY3_DIR = DATA_DIR / "results_day3"
RESULTS_DAY4_DIR = DATA_DIR / "results_day4"
RESULTS_DAY4_DIR.mkdir(parents=True, exist_ok=True)

print("Using data dir:", DATA_DIR)
print("Using figures dir:", FIG_DIR)
print("Using day3 results dir:", RESULTS_DAY3_DIR)
print("Using day4 results dir:", RESULTS_DAY4_DIR)

Using data dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:
Using figures dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/figures:
Using day3 results dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:/results_day3
Using day4 results dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:/results_day4


In [2]:
def add_volatility(df, window=10):
    df = df.sort_values("Date").copy()
    df["log_return"] = np.log(df["Close"]).diff()
    df["rv_10"] = df["log_return"].rolling(window).std()
    return df


def make_target(df):
    df = df.sort_values("Date").copy()
    df["target_rv_10"] = df["rv_10"].shift(-1)
    return df


def add_features(df, n_return_lags=5):
    df = df.sort_values("Date").copy()
    df["log_close"] = np.log(df["Close"])

    for k in range(1, n_return_lags + 1):
        df[f"log_return_lag_{k}"] = df["log_return"].shift(k)

    df["rv_10_lag_1"] = df["rv_10"].shift(1)
    df["log_volume"] = np.log(df["Volume"].replace(0, np.nan))
    df["volume_change"] = df["Volume"].pct_change()
    return df


def build_day2_ready_df(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None)
    df = make_target(add_volatility(df, window=10))
    df = add_features(df, n_return_lags=5)
    return df.dropna().sort_values("Date").reset_index(drop=True)


aapl = build_day2_ready_df(DATA_DIR / "AAPL_clean.csv")
msft = build_day2_ready_df(DATA_DIR / "MSFT_clean.csv")

feature_cols = [
    c
    for c in aapl.columns
    if c.startswith("log_return_lag_")
    or c in ["log_close", "rv_10", "rv_10_lag_1", "log_volume", "volume_change"]
]
target_col = "target_rv_10"


def time_split(df, train_end, val_end):
    train = df[df["Date"] <= train_end].copy()
    val = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
    test = df[df["Date"] > val_end].copy()
    return train, val, test


train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2022-12-31")

a_train, a_val, a_test = time_split(aapl, train_end, val_end)
m_train, m_val, m_test = time_split(msft, train_end, val_end)

print("Feature columns:", feature_cols)
print("AAPL split sizes:", len(a_train), len(a_val), len(a_test))
print("MSFT split sizes:", len(m_train), len(m_val), len(m_test))

Feature columns: ['rv_10', 'log_close', 'log_return_lag_1', 'log_return_lag_2', 'log_return_lag_3', 'log_return_lag_4', 'log_return_lag_5', 'rv_10_lag_1', 'log_volume', 'volume_change']
AAPL split sizes: 764 252 228
MSFT split sizes: 764 252 228


In [3]:
def eval_forecasts(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse


def fit_and_eval(model, train, val, test, label):
    X_train = train[feature_cols].values
    y_train = train[target_col].values
    X_val = val[feature_cols].values
    y_val = val[target_col].values
    X_test = test[feature_cols].values
    y_test = test[target_col].values

    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    mae_val, rmse_val = eval_forecasts(y_val, y_val_pred)
    mae_test, rmse_test = eval_forecasts(y_test, y_test_pred)

    print(
        f"{label}: Val MAE={mae_val:.5f}, RMSE={rmse_val:.5f} | "
        f"Test MAE={mae_test:.5f}, RMSE={rmse_test:.5f}"
    )

    test_out = test.copy()
    test_out["pred"] = y_test_pred
    return model, test_out, mae_val, rmse_val, mae_test, rmse_test


def tune_and_fit(make_model, param_grid, train, val, test, label):
    best = None

    for params in param_grid:
        model = make_model(params)
        _, test_out, mae_val, rmse_val, mae_test, rmse_test = fit_and_eval(
            model, train, val, test, f"{label} params={params}"
        )

        row = {
            "Ticker": label.split()[0],
            "Model": label.split(" ", 1)[1],
            "Params": str(params),
            "Val_MAE": mae_val,
            "Val_RMSE": rmse_val,
            "Test_MAE": mae_test,
            "Test_RMSE": rmse_test,
            "test_df": test_out,
            "model_obj": model,
        }

        if (best is None) or (row["Val_RMSE"] < best["Val_RMSE"]):
            best = row

    return best

In [4]:
tree_grid = [
    {"max_depth": 3, "min_samples_leaf": 10, "random_state": 0},
    {"max_depth": 5, "min_samples_leaf": 20, "random_state": 0},
    {"max_depth": 8, "min_samples_leaf": 30, "random_state": 0},
]

rf_grid = [
    {"n_estimators": 150, "max_depth": None, "min_samples_leaf": 10, "n_jobs": -1, "random_state": 0},
    {"n_estimators": 250, "max_depth": None, "min_samples_leaf": 10, "n_jobs": -1, "random_state": 0},
    {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 15, "n_jobs": -1, "random_state": 0},
]

gb_grid = [
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 20, "random_state": 0},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 20, "random_state": 0},
    {"n_estimators": 250, "learning_rate": 0.10, "max_depth": 2, "min_samples_leaf": 20, "random_state": 0},
]


a_tree_best = tune_and_fit(lambda p: DecisionTreeRegressor(**p), tree_grid, a_train, a_val, a_test, "AAPL DecisionTree")
a_rf_best = tune_and_fit(lambda p: RandomForestRegressor(**p), rf_grid, a_train, a_val, a_test, "AAPL RandomForest")
a_gb_best = tune_and_fit(lambda p: GradientBoostingRegressor(**p), gb_grid, a_train, a_val, a_test, "AAPL GradientBoosting")

m_tree_best = tune_and_fit(lambda p: DecisionTreeRegressor(**p), tree_grid, m_train, m_val, m_test, "MSFT DecisionTree")
m_rf_best = tune_and_fit(lambda p: RandomForestRegressor(**p), rf_grid, m_train, m_val, m_test, "MSFT RandomForest")
m_gb_best = tune_and_fit(lambda p: GradientBoostingRegressor(**p), gb_grid, m_train, m_val, m_test, "MSFT GradientBoosting")

best_rows = [a_tree_best, a_rf_best, a_gb_best, m_tree_best, m_rf_best, m_gb_best]
best_summary = pd.DataFrame([
    {
        "Ticker": r["Ticker"],
        "Model": r["Model"],
        "Params": r["Params"],
        "Val_MAE": r["Val_MAE"],
        "Val_RMSE": r["Val_RMSE"],
        "Test_MAE": r["Test_MAE"],
        "Test_RMSE": r["Test_RMSE"],
    }
    for r in best_rows
]).sort_values(["Ticker", "Val_RMSE"])

best_summary

AAPL DecisionTree params={'max_depth': 3, 'min_samples_leaf': 10, 'random_state': 0}: Val MAE=0.00214, RMSE=0.00296 | Test MAE=0.00183, RMSE=0.00231
AAPL DecisionTree params={'max_depth': 5, 'min_samples_leaf': 20, 'random_state': 0}: Val MAE=0.00193, RMSE=0.00284 | Test MAE=0.00133, RMSE=0.00190
AAPL DecisionTree params={'max_depth': 8, 'min_samples_leaf': 30, 'random_state': 0}: Val MAE=0.00204, RMSE=0.00312 | Test MAE=0.00117, RMSE=0.00176
AAPL RandomForest params={'n_estimators': 150, 'max_depth': None, 'min_samples_leaf': 10, 'n_jobs': -1, 'random_state': 0}: Val MAE=0.00185, RMSE=0.00275 | Test MAE=0.00109, RMSE=0.00167
AAPL RandomForest params={'n_estimators': 250, 'max_depth': None, 'min_samples_leaf': 10, 'n_jobs': -1, 'random_state': 0}: Val MAE=0.00184, RMSE=0.00275 | Test MAE=0.00107, RMSE=0.00166
AAPL RandomForest params={'n_estimators': 200, 'max_depth': 8, 'min_samples_leaf': 15, 'n_jobs': -1, 'random_state': 0}: Val MAE=0.00177, RMSE=0.00268 | Test MAE=0.00108, RMSE=0.0

,Ticker,Model,Params,Val_MAE,Val_RMSE,Test_MAE,Test_RMSE
1,AAPL,RandomForest,"{'n_estimators': 200, 'max_depth': 8, 'min_sam...",0.001768,0.002684,0.001077,0.001680
2,AAPL,GradientBoosting,"{'n_estimators': 250, 'learning_rate': 0.1, 'm...",0.001969,0.002822,0.001164,0.001723
0,AAPL,DecisionTree,"{'max_depth': 5, 'min_samples_leaf': 20, 'rand...",0.001931,0.002842,0.001326,0.001901
4,MSFT,RandomForest,"{'n_estimators': 250, 'max_depth': None, 'min_...",0.001865,0.002890,0.001219,0.001994
3,MSFT,DecisionTree,"{'max_depth': 5, 'min_samples_leaf': 20, 'rand...",0.002024,0.002961,0.001411,0.002196
5,MSFT,GradientBoosting,"{'n_estimators': 250, 'learning_rate': 0.1, 'm...",0.002457,0.003357,0.001399,0.002127


In [5]:
# Pull in Day 3 baseline + linear metrics for full comparison table.
metrics_day3 = pd.read_csv(RESULTS_DAY3_DIR / "metrics_day3.csv")

# Keep test metrics only for direct model comparison.
day3_test = metrics_day3[metrics_day3["Split"] == "Test"].copy()
day3_test = day3_test.rename(columns={"MAE": "Test_MAE", "RMSE": "Test_RMSE"})
day3_test = day3_test[["Ticker", "Model", "Test_MAE", "Test_RMSE"]]

# Day 4 best models test metrics.
day4_test = best_summary[["Ticker", "Model", "Test_MAE", "Test_RMSE"]].copy()

comparison_test = pd.concat([day3_test, day4_test], ignore_index=True)
comparison_test = comparison_test.sort_values(["Ticker", "Test_RMSE"]).reset_index(drop=True)

comparison_test

,Ticker,Model,Test_MAE,Test_RMSE
0,AAPL,Naive,0.000956,0.001670
1,AAPL,RandomForest,0.001077,0.001680
2,AAPL,GradientBoosting,0.001164,0.001723
3,AAPL,LinearRegression,0.001231,0.001792
4,AAPL,DecisionTree,0.001326,0.001901
5,AAPL,RollingMean10,0.002485,0.003306
6,MSFT,RandomForest,0.001219,0.001994
7,MSFT,LinearRegression,0.001310,0.002024
8,MSFT,Naive,0.001195,0.002069
9,MSFT,GradientBoosting,0.001399,0.002127


In [6]:
def plot_importances(model, feature_names, name, top_k=10):
    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1][:top_k]

    plt.figure(figsize=(9, 4))
    plt.bar(range(len(idx)), importances[idx])
    plt.xticks(range(len(idx)), [feature_names[i] for i in idx], rotation=45, ha="right")
    plt.title(f"{name} Feature Importances")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}_feature_importance.png", dpi=150)
    plt.close()


plot_importances(a_rf_best["model_obj"], feature_cols, "AAPL_RF")
plot_importances(m_rf_best["model_obj"], feature_cols, "MSFT_RF")

print("Saved feature importance plots:")
print(" -", (FIG_DIR / "AAPL_RF_feature_importance.png").name)
print(" -", (FIG_DIR / "MSFT_RF_feature_importance.png").name)

Saved feature importance plots:
 - AAPL_RF_feature_importance.png
 - MSFT_RF_feature_importance.png


In [7]:
a_linear_test = pd.read_csv(RESULTS_DAY3_DIR / "AAPL_test_with_predictions.csv", parse_dates=["Date"])
m_linear_test = pd.read_csv(RESULTS_DAY3_DIR / "MSFT_test_with_predictions.csv", parse_dates=["Date"])


def plot_model_comparison(tree_test_df, linear_df, ticker, model_label):
    plt.figure(figsize=(10, 4))
    plt.plot(tree_test_df["Date"], tree_test_df[target_col], label="Realized RV", linewidth=1.8)
    plt.plot(tree_test_df["Date"], tree_test_df["pred"], label=model_label, alpha=0.85)
    plt.plot(linear_df["Date"], linear_df["pred_linear"], label="Linear", alpha=0.75)
    plt.xlabel("Date")
    plt.ylabel("Volatility")
    plt.title(f"{ticker} - {model_label} vs Linear vs Realized")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{ticker}_rf_vs_linear.png", dpi=150)
    plt.close()


plot_model_comparison(a_rf_best["test_df"], a_linear_test, "AAPL", "Random Forest")
plot_model_comparison(m_rf_best["test_df"], m_linear_test, "MSFT", "Random Forest")

print("Saved comparison plots:")
print(" -", (FIG_DIR / "AAPL_rf_vs_linear.png").name)
print(" -", (FIG_DIR / "MSFT_rf_vs_linear.png").name)

Saved comparison plots:
 - AAPL_rf_vs_linear.png
 - MSFT_rf_vs_linear.png


In [8]:
# Save all Day 4 outputs
best_summary.to_csv(RESULTS_DAY4_DIR / "tree_model_tuning_summary.csv", index=False)
comparison_test.to_csv(RESULTS_DAY4_DIR / "metrics_day4_comparison_test.csv", index=False)

a_tree_best["test_df"].to_csv(RESULTS_DAY4_DIR / "AAPL_decision_tree_test_predictions.csv", index=False)
a_rf_best["test_df"].to_csv(RESULTS_DAY4_DIR / "AAPL_random_forest_test_predictions.csv", index=False)
a_gb_best["test_df"].to_csv(RESULTS_DAY4_DIR / "AAPL_gradient_boosting_test_predictions.csv", index=False)

m_tree_best["test_df"].to_csv(RESULTS_DAY4_DIR / "MSFT_decision_tree_test_predictions.csv", index=False)
m_rf_best["test_df"].to_csv(RESULTS_DAY4_DIR / "MSFT_random_forest_test_predictions.csv", index=False)
m_gb_best["test_df"].to_csv(RESULTS_DAY4_DIR / "MSFT_gradient_boosting_test_predictions.csv", index=False)

print("Saved Day 4 files to:", RESULTS_DAY4_DIR)
for p in sorted(RESULTS_DAY4_DIR.glob("*.csv")):
    print(" -", p.name)

Saved Day 4 files to: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:/results_day4
 - AAPL_decision_tree_test_predictions.csv
 - AAPL_gradient_boosting_test_predictions.csv
 - AAPL_random_forest_test_predictions.csv
 - MSFT_decision_tree_test_predictions.csv
 - MSFT_gradient_boosting_test_predictions.csv
 - MSFT_random_forest_test_predictions.csv
 - metrics_day4_comparison_test.csv
 - tree_model_tuning_summary.csv


### Day 4 Checklist (Implemented)

- Decision Tree, Random Forest, and Gradient Boosting models trained for AAPL/MSFT.
- Light tuning via validation RMSE and best-model selection per ticker/model family.
- Comparison table created (Day 3 baselines + linear + Day 4 tree models).
- RF feature importance plots and RF vs Linear vs Realized plots saved to `figures:`.
- Model metrics and test predictions saved to `data:/results_day4`.

In [9]:
# Paper-ready metrics table: fixed model order, sorted rows, rounded values.
paper_model_order = [
    "Naive",
    "RollingMean10",
    "LinearRegression",
    "DecisionTree",
    "RandomForest",
    "GradientBoosting",
]

paper_table = pd.read_csv(RESULTS_DAY4_DIR / "metrics_day4_comparison_test.csv")
paper_table["Model"] = pd.Categorical(
    paper_table["Model"], categories=paper_model_order, ordered=True
)

paper_table = (
    paper_table.sort_values(["Ticker", "Model"])
    .reset_index(drop=True)
    .copy()
)
paper_table["MAE"] = paper_table["Test_MAE"].round(5)
paper_table["RMSE"] = paper_table["Test_RMSE"].round(5)

paper_table = paper_table[["Ticker", "Model", "MAE", "RMSE"]]
paper_table.to_csv(RESULTS_DAY4_DIR / "metrics_day4_paper_ready.csv", index=False)
paper_table

,Ticker,Model,MAE,RMSE
0,AAPL,Naive,0.00096,0.00167
1,AAPL,RollingMean10,0.00248,0.00331
2,AAPL,LinearRegression,0.00123,0.00179
3,AAPL,DecisionTree,0.00133,0.00190
4,AAPL,RandomForest,0.00108,0.00168
5,AAPL,GradientBoosting,0.00116,0.00172
6,MSFT,Naive,0.00119,0.00207
7,MSFT,RollingMean10,0.00338,0.00433
8,MSFT,LinearRegression,0.00131,0.00202
9,MSFT,DecisionTree,0.00141,0.00220
